# 06 — Customer Segmentation & Profiling
## K-Means Clustering on Customer Financial Behavior
Goal: Group customers into 5 distinct behavioral segments based on credit limit,
transaction volume, and activity patterns to enable targeted marketing.

In [1]:
import pandas as pd, numpy as np, sqlite3
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load Data ─────────────────────────────────────────────────
df = pd.read_csv('../../data/processed/customer_clean.csv')
print(f"Loaded: {df.shape}")

Loaded: (10127, 22)


In [3]:
# ── Feature Selection & Scaling ───────────────────────────────────
seg_features = ['Credit_Limit', 'Total_Trans_Amt', 'Total_Trans_Ct',
                'Avg_Utilization_Ratio', 'Months_Inactive_12_mon']
X = df[seg_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Features scaled. Stats:")
print(pd.DataFrame(X_scaled, columns=seg_features).describe().round(3))

Features scaled. Stats:
       Credit_Limit  Total_Trans_Amt  Total_Trans_Ct  Avg_Utilization_Ratio  \
count    10127.0000       10127.0000      10127.0000             10127.0000   
mean         0.0000          -0.0000         -0.0000                 0.0000   
std          1.0000           1.0000          1.0000                 1.0000   
min         -0.7920          -1.1460         -2.3370                -0.9970   
25%         -0.6690          -0.6620         -0.8460                -0.9140   
50%         -0.4490          -0.1490          0.0910                -0.3590   
75%          0.2680           0.0990          0.6880                 0.8270   
max          2.8480           4.1450          3.1590                 2.6270   

       Months_Inactive_12_mon  
count              10127.0000  
mean                   0.0000  
std                    1.0000  
min                   -2.3170  
25%                   -0.3380  
50%                   -0.3380  
75%                    0.6520  
max     

In [4]:
# ── Elbow Method & Silhouette Score ───────────────────────────────
# This will take a few seconds...
inertias, sil_scores = [], []
K_range = range(2, 11)
print("Evaluating K=2 to 10...")
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f"k={k}: Inertia={km.inertia_:.0f}, Silhouette={sil_scores[-1]:.4f}")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, name="Inertia (Elbow)", mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=list(K_range), y=sil_scores, name="Silhouette Score", mode="lines+markers", marker=dict(color='red')), secondary_y=True)
fig.update_layout(title_text="K-Means Optimization: Elbow Curve vs Silhouette Score", template='plotly_white')
fig.update_xaxes(title_text="Number of Clusters (k)")
fig.update_yaxes(title_text="Inertia", secondary_y=False)
fig.update_yaxes(title_text="Silhouette Score", secondary_y=True)
fig.show()

Evaluating K=2 to 10...


k=2: Inertia=38705, Silhouette=0.3503


k=3: Inertia=29359, Silhouette=0.2701


k=4: Inertia=24565, Silhouette=0.2741


k=5: Inertia=20926, Silhouette=0.2658


k=6: Inertia=18263, Silhouette=0.2702


k=7: Inertia=16450, Silhouette=0.2764


k=8: Inertia=15065, Silhouette=0.2790


k=9: Inertia=13928, Silhouette=0.2803


k=10: Inertia=12750, Silhouette=0.2848


In [5]:
# ── Fit Final Model (k=5) & Label Clusters ──────────────────────────
km_final = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_scaled)

profile = df.groupby('cluster').agg(
    count=('Credit_Limit','count'),
    avg_credit_limit=('Credit_Limit','mean'),
    avg_trans_amt=('Total_Trans_Amt','mean'),
    avg_trans_ct=('Total_Trans_Ct','mean'),
    avg_utilization=('Avg_Utilization_Ratio','mean'),
    avg_months_inactive=('Months_Inactive_12_mon','mean')
).round(2)

churn_by_cluster = df.groupby('cluster')['is_attrited'].mean().round(4) * 100
profile['churn_pct'] = churn_by_cluster
print("Cluster profiles (REAL values):")
print(profile.to_string())

# Dynamic labeling
segment_map = {}
remaining = list(profile.index)

premium = profile.loc[remaining, 'avg_credit_limit'].idxmax()
segment_map[premium] = 'Premium Customers'
remaining.remove(premium)

daily = profile.loc[remaining, 'avg_trans_ct'].idxmax()
segment_map[daily] = 'Daily Spenders'
remaining.remove(daily)

atrisk = profile.loc[remaining, 'avg_months_inactive'].idxmax()
segment_map[atrisk] = 'At-Risk Customers'
remaining.remove(atrisk)

dealh = profile.loc[remaining, 'avg_trans_amt'].idxmax()
segment_map[dealh] = 'Deal Hunters'
remaining.remove(dealh)

segment_map[remaining[0]] = 'Silent Users'

df['segment'] = df['cluster'].map(segment_map)
profile['segment'] = profile.index.map(segment_map)
print("\nSegment distribution:")
print(df['segment'].value_counts())

Cluster profiles (REAL values):
         count  avg_credit_limit  avg_trans_amt  avg_trans_ct  avg_utilization  avg_months_inactive  churn_pct
cluster                                                                                                       
0         2730         6769.5600      3372.6300       59.3700           0.1400               1.5500    15.6800
1         2451         6464.1800      3324.8200       57.5100           0.1400               3.3600    29.8700
2          869        13948.5300     13834.3800      108.4400           0.1700               2.2200     1.7300
3         1194        27970.8000      3798.5500       60.6700           0.0400               2.3100    17.5000
4         2883         2626.7000      3706.6400       64.9000           0.6500               2.2800     8.4300

Segment distribution:
segment
Deal Hunters         2883
Silent Users         2730
At-Risk Customers    2451
Premium Customers    1194
Daily Spenders        869
Name: count, dtype: int64


In [6]:
# ── PCA Visualization ─────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
df['pca_x'] = coords[:,0]
df['pca_y'] = coords[:,1]
var_explained = pca.explained_variance_ratio_.sum()*100
print(f"PCA explained variance: {var_explained:.1f}%")
fig_pca = px.scatter(df, x='pca_x', y='pca_y', color='segment', opacity=0.6,
                     title=f"Customer Segments in PCA Space (explains {var_explained:.1f}% of variance)")
fig_pca.update_layout(template='plotly_white')
fig_pca.show()

PCA explained variance: 66.1%


In [7]:
# ── SQL Cross-Analysis ────────────────────────────────────────
con = sqlite3.connect(':memory:')
df.to_sql('seg', con, index=False, if_exists='replace')

q1 = pd.read_sql_query("""
    SELECT segment, COUNT(*) as count,
           ROUND(AVG(Credit_Limit),2) as avg_credit,
           ROUND(AVG(Total_Trans_Amt),2) as avg_trans_amt,
           ROUND(AVG(Total_Trans_Ct),2) as avg_trans_ct,
           ROUND(AVG(Avg_Utilization_Ratio),4) as avg_util,
           ROUND(AVG(Months_Inactive_12_mon),2) as avg_inactive,
           ROUND(AVG(is_attrited)*100,2) as churn_pct
    FROM seg GROUP BY segment ORDER BY avg_credit DESC
""", con)
print("Q1: Segment Profile Full")
print(q1.to_string(index=False))

q2 = pd.read_sql_query("""
    SELECT segment, Card_Category, COUNT(*) as count
    FROM seg GROUP BY segment, Card_Category
""", con)

q3 = pd.read_sql_query("""
    SELECT segment, Income_Category, COUNT(*) as count
    FROM seg GROUP BY segment, Income_Category
""", con)

Q1: Segment Profile Full
          segment  count  avg_credit  avg_trans_amt  avg_trans_ct  avg_util  avg_inactive  churn_pct
Premium Customers   1194  27970.8000      3798.5500       60.6700    0.0432        2.3100    17.5000
   Daily Spenders    869  13948.5300     13834.3800      108.4400    0.1745        2.2200     1.7300
     Silent Users   2730   6769.5600      3372.6300       59.3700    0.1366        1.5500    15.6800
At-Risk Customers   2451   6464.1800      3324.8200       57.5100    0.1389        3.3600    29.8700
     Deal Hunters   2883   2626.7000      3706.6400       64.9000    0.6477        2.2800     8.4300


In [8]:
# ── Visualizations ────────────────────────────────────────────
fig_donut = px.pie(q1, names='segment', values='count', hole=0.4, title="Segment Distribution")
fig_donut.update_layout(template='plotly_white')
fig_donut.show()

fig_churn = px.bar(q1.sort_values('churn_pct'), x='churn_pct', y='segment', orientation='h', title="Churn Rate by Segment (%)")
fig_churn.update_layout(template='plotly_white')
fig_churn.show()

scaler_mm = MinMaxScaler()
prof_features = ['avg_credit', 'avg_trans_amt', 'avg_trans_ct', 'avg_util', 'avg_inactive']
q1_scaled = q1.copy()
q1_scaled[prof_features] = scaler_mm.fit_transform(q1[prof_features])
fig_radar = go.Figure()
for i, row in q1_scaled.iterrows():
    fig_radar.add_trace(go.Scatterpolar(r=row[prof_features].values.tolist() + [row[prof_features].values[0]],
                                        theta=prof_features + [prof_features[0]],
                                        name=row['segment']))
fig_radar.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])), showlegend=True, title="Segment Profiles (Normalized)", template='plotly_white')
fig_radar.show()

card_pivot = q2.pivot(index='segment', columns='Card_Category', values='count').fillna(0)
fig_heat = px.imshow(card_pivot, title="Segment x Card Category Heatmap", text_auto=True, aspect="auto")
fig_heat.show()

fig_box = px.box(df, x='segment', y='Credit_Limit', color='segment', title="Credit Limit by Segment")
fig_box.update_layout(template='plotly_white')
fig_box.show()

fig_violin = px.violin(df, x='segment', y='Total_Trans_Amt', color='segment', box=True, title="Transaction Amount by Segment")
fig_violin.update_layout(template='plotly_white')
fig_violin.show()

In [9]:
# ── Save Output ────────────────────────────────────────────
import joblib, os
os.makedirs('../../models/segmentation', exist_ok=True)
joblib.dump(km_final, '../../models/segmentation/kmeans_model.pkl')
joblib.dump(scaler, '../../models/segmentation/scaler.pkl')
joblib.dump(segment_map, '../../models/segmentation/segment_map.pkl')
df.to_csv('../../data/processed/customer_with_segments.csv', index=False)
profile.to_csv('../../data/processed/segment_profiles.csv')
print("Saved model, scaler, segment_map, and customer_with_segments.csv")
con.close()

Saved model, scaler, segment_map, and customer_with_segments.csv
